# Template — Relatório de Entrada SAGI

Este notebook lê arquivos `.xls` exportados pelo sistema SAGI (formato XML SpreadsheetML gerado pelo FastReport),  
extrai os registros de movimentação e devolve um `DataFrame` limpo e tipado para análise.

**Como usar:** altere apenas a variável `ARQUIVO` na célula abaixo e execute todas as células.

In [ ]:
# =============================================================
# CONFIGURAÇÃO — altere apenas esta variável
# =============================================================
ARQUIVO = '02-Rferencias/sagi_MIUDA.xls'

# =============================================================
# Leitura do arquivo XML
# =============================================================
import xml.etree.ElementTree as ET

NS   = 'urn:schemas-microsoft-com:office:spreadsheet'
ns   = {'ss': NS}
ATTR = lambda name: f'{{{NS}}}{name}'

tree = ET.parse(ARQUIVO)
rows = tree.findall('.//ss:Row', ns)

print(f'Arquivo lido: {ARQUIVO}')
print(f'Total de linhas XML encontradas: {len(rows)}')

Arquivo lido: sagi_MIUDA.xls
Total de linhas XML encontradas: 7304


In [3]:
import pandas as pd
import re

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def get_cell(row, idx):
    """Retorna o texto da célula com ss:Index == idx, ou '' se ausente."""
    for cell in row.findall('ss:Cell', ns):
        if cell.get(ATTR('Index')) == str(idx):
            data = cell.find('ss:Data', ns)
            return (data.text or '').strip() if data is not None else ''
    return ''

def is_data_row(row):
    """
    Linha de dados principal: a célula idx=1 existe e contém
    um número inteiro (número do boleto).
    """
    val = get_cell(row, 1)
    return bool(val and re.fullmatch(r'\d+', val))

def split_fornecedor(text):
    """Separa '9731-JEAN CARLOS MARTINS' em (cod, nome)."""
    m = re.match(r'^(\d+)-(.+)$', text.strip())
    if m:
        return m.group(1), m.group(2).strip()
    return '', text.strip()

# ------------------------------------------------------------------
# Extração
# ------------------------------------------------------------------

records = []
i = 0

while i < len(rows):
    row = rows[i]

    if not is_data_row(row):
        i += 1
        continue

    # --- campos da linha principal ---
    boleto       = get_cell(row, 1)
    nota_fiscal  = get_cell(row, 6)
    data         = get_cell(row, 10)
    fornecedor   = get_cell(row, 14)
    quantidade   = get_cell(row, 20)
    preco        = get_cell(row, 22)
    total_rs     = get_cell(row, 29)
    total_icms   = get_cell(row, 34)
    placa        = get_cell(row, 38)
    produto      = get_cell(row, 39)
    un           = get_cell(row, 41)
    valor_ipi    = get_cell(row, 44)
    desconto     = get_cell(row, 51)
    desconto_rs  = get_cell(row, 52)
    frete        = get_cell(row, 56)
    pesagem      = get_cell(row, 61)

    fornecedor_cod, fornecedor_nome = split_fornecedor(fornecedor)

    # Limpar 'R$ 0,00' do campo valor_ipi -> manter só o número
    valor_ipi = re.sub(r'R\$\s*', '', valor_ipi).replace(',', '.')

    # --- classificador: próxima linha não vazia com idx=63 ---
    classificador = ''
    for j in range(i + 1, min(i + 5, len(rows))):
        val = get_cell(rows[j], 63)
        if val:
            classificador = val
            break

    # --- data de pagamento e CPF/CNPJ: buscados nas ~8 linhas seguintes ---
    data_pagamento = ''
    cpf_cnpj       = ''
    for j in range(i + 1, min(i + 10, len(rows))):
        r = rows[j]

        v1 = get_cell(r, 1)
        if v1.startswith('Data Pagamento:'):
            data_pagamento = v1.replace('Data Pagamento:', '').strip()

        v15 = get_cell(r, 15)
        if v15.startswith('CPF / CNPJ:'):
            cpf_cnpj = v15.replace('CPF / CNPJ:', '').strip()

        if j > i + 1 and is_data_row(r):
            break

    records.append({
        'boleto'          : boleto,
        'nota_fiscal'     : nota_fiscal,
        'data'            : data,
        'fornecedor_cod'  : fornecedor_cod,
        'fornecedor_nome' : fornecedor_nome,
        'quantidade'      : quantidade,
        'preco'           : preco,
        'total_rs'        : total_rs,
        'total_icms'      : total_icms,
        'placa'           : placa,
        'produto'         : produto,
        'un'              : un,
        'valor_ipi'       : valor_ipi,
        'desconto'        : desconto,
        'desconto_rs'     : desconto_rs,
        'frete'           : frete,
        'pesagem'         : pesagem,
        'classificador'   : classificador,
        'data_pagamento'  : data_pagamento,
        'cpf_cnpj'        : cpf_cnpj,
    })

    i += 1

df_raw = pd.DataFrame(records)
print(f'Registros extraídos: {len(df_raw)}')
df_raw.head(3)

Registros extraídos: 789


,boleto,nota_fiscal,data,fornecedor_cod,fornecedor_nome,quantidade,preco,total_rs,total_icms,placa,produto,un,valor_ipi,desconto,desconto_rs,frete,pesagem,classificador,data_pagamento,cpf_cnpj
0,294490,38128-1,02/03/2026,9731,JEAN CARLOS MARTINS,2510.000,0.90000,2259.00,2259.00,GPZ-5F48,SFMIU-S SUCATA DE FERRO (MIUDA),KG,0.00,0.0,0.00,EMPRESA,ELETRÔNICA,VALTER VIERIA,05/03/2026,047.854.051-55
1,294495,19355-2,02/03/2026,15189,LORENE DA SILVA AMORIM,390.000,0.80000,312.00,312.00,HQR-3159,SFMIU-S SUCATA DE FERRO (MIUDA),KG,0.00,0.0,0.00,FORNECEDOR,ELETRÔNICA,LUIZ GUILHERME,02/03/2026,446.027.118-47
2,294491,1104-1,02/03/2026,5526,CIRCO NEGRETE GARCIA,70.000,0.80000,56.00,56.00,BIU-9848,SFMIU-S SUCATA DE FERRO (MIUDA),KG,0.00,0.0,0.00,FORNECEDOR,ELETRÔNICA,ALEKSANDER APARECIDO GONÇALVES SILVA,02/03/2026,354.105.159-00


In [4]:
# =============================================================
# Tipagem e formatação
# =============================================================

df = df_raw.copy()

# --- colunas de texto: strip e substituição de múltiplos espaços ---
text_cols = ['boleto', 'nota_fiscal', 'fornecedor_cod', 'fornecedor_nome',
             'placa', 'produto', 'un', 'frete', 'pesagem', 'classificador', 'cpf_cnpj']
for col in text_cols:
    df[col] = df[col].str.strip().str.replace(r'\s+', ' ', regex=True)

# --- datas: tipo datetime64, componente de hora zerado ---
df['data']           = pd.to_datetime(df['data'],           format='%d/%m/%Y', errors='coerce').dt.normalize()
df['data_pagamento'] = pd.to_datetime(df['data_pagamento'], format='%d/%m/%Y', errors='coerce').dt.normalize()

# --- numéricas: ponto é o separador decimal no relatório SAGI ---
num_cols = ['quantidade', 'preco', 'total_rs', 'total_icms', 'valor_ipi', 'desconto', 'desconto_rs']
for col in num_cols:
    df[col] = (
        df[col]
        .str.replace(r'[^\d.\-]', '', regex=True)  # remove qualquer caractere não-numérico restante
        .replace('', pd.NA)
        .astype('Float64')
    )

# --- ordenar por data e boleto para facilitar análise ---
df = df.sort_values(['data', 'boleto']).reset_index(drop=True)

print('Tipos de dados:')
print(df.dtypes)
print(f'\nPeriodo: {df["data"].min().strftime("%d/%m/%Y")} a {df["data"].max().strftime("%d/%m/%Y")}')
print(f'Total de registros: {len(df)}')

Tipos de dados:
boleto                        str
nota_fiscal                   str
data               datetime64[us]
fornecedor_cod                str
fornecedor_nome               str
quantidade                Float64
preco                     Float64
total_rs                  Float64
total_icms                Float64
placa                         str
produto                       str
un                            str
valor_ipi                 Float64
desconto                  Float64
desconto_rs               Float64
frete                         str
pesagem                       str
classificador                 str
data_pagamento     datetime64[us]
cpf_cnpj                      str
dtype: object

Periodo: 02/03/2026 a 26/03/2026
Total de registros: 789


In [ ]:
# =============================================================
# Visualização e validação
# =============================================================

print('=== Informações gerais ===')
df.info()

print('\n=== Primeiros registros ===')

# Formata colunas de data como dd/MM/yyyy somente na exibição
# O tipo datetime64 é preservado no df para análise
date_fmt = lambda x: x.strftime('%d/%m/%Y') if pd.notna(x) else ''

df.head(10).style.format({
    'data'           : date_fmt,
    'data_pagamento' : date_fmt,
})

# Para salvar o DataFrame `df` em um arquivo Excel:
df.to_excel('relatorio_entrada.xlsx', index=False)

# Ou para salvar em CSV:
#df.to_csv('relatorio_entrada.csv', index=False)

=== Informações gerais ===
<class 'pandas.DataFrame'>
RangeIndex: 789 entries, 0 to 788
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   boleto           789 non-null    str           
 1   nota_fiscal      789 non-null    str           
 2   data             789 non-null    datetime64[us]
 3   fornecedor_cod   789 non-null    str           
 4   fornecedor_nome  789 non-null    str           
 5   quantidade       789 non-null    Float64       
 6   preco            789 non-null    Float64       
 7   total_rs         789 non-null    Float64       
 8   total_icms       789 non-null    Float64       
 9   placa            789 non-null    str           
 10  produto          789 non-null    str           
 11  un               789 non-null    str           
 12  valor_ipi        789 non-null    Float64       
 13  desconto         789 non-null    Float64       
 14  desconto_rs      789 non-n